In [1]:
# Cell 1: Imports and setup
import numpy as np
import pandas as pd
import scipy.sparse as sps
from sklearn.neighbors import KNeighborsTransformer
from sklearn.metrics import pairwise_distances

from skclust.kneighbors import (
    kneighbors_graph_from_transformer,
    brute_force_kneighbors_graph_from_rectangular_distance,
    pairwise_distances_kneighbors,
    convert_distance_matrix_to_kneighbors_matrix,
    kneighbors_to_igraph,
    KNeighborsCosineSimilarity,
)

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All imports successful")

✓ All imports successful


/Users/josh/miniforge3/envs/torch/lib/python3.12/site-packages/skclust/hierarchical.py:42: UserWarning: fastcluster not available, using scipy.cluster.hierarchy.linkage
  warnings.warn("fastcluster not available, using scipy.cluster.hierarchy.linkage")
/Users/josh/miniforge3/envs/torch/lib/python3.12/site-packages/skclust/hierarchical.py:49: UserWarning: skbio not available, tree functionality will be limited
  warnings.warn("skbio not available, tree functionality will be limited")


In [2]:
# Cell 2: Test kneighbors_graph_from_transformer - Basic functionality
print("Testing kneighbors_graph_from_transformer...")

X = np.random.randn(10, 5)
n_neighbors = 3

# Test 1: Using class with connectivity mode
graph = kneighbors_graph_from_transformer(
    X,
    knn_transformer=KNeighborsTransformer,
    n_neighbors=n_neighbors,
    mode='connectivity',
    include_self=False
)

assert isinstance(graph, sps.csr_matrix), "Should return sparse matrix"
assert graph.shape == (10, 10), "Shape should be (10, 10)"
assert np.all(np.array(graph.sum(axis=1)).ravel() == n_neighbors), \
    "Each row should have exactly n_neighbors non-zero entries"
print("  ✓ Class with connectivity mode")

# Test 2: Using class with distance mode
graph = kneighbors_graph_from_transformer(
    X,
    knn_transformer=KNeighborsTransformer,
    n_neighbors=n_neighbors,
    mode='distance',
    include_self=False
)

assert isinstance(graph, sps.csr_matrix), "Should return sparse matrix"
assert np.any(graph.data != 1.0), "Should have non-binary distance values"
print("  ✓ Class with distance mode")

# Test 3: Using fitted transformer
knn = KNeighborsTransformer(n_neighbors=n_neighbors, mode='distance')
knn.fit(X)

graph = kneighbors_graph_from_transformer(
    X,
    knn_transformer=knn,
    mode='distance',
    include_self=False
)

assert isinstance(graph, sps.csr_matrix), "Should return sparse matrix"
print("  ✓ Fitted transformer")

# Test 4: include_self=True
graph = kneighbors_graph_from_transformer(
    X,
    knn_transformer=KNeighborsTransformer,
    n_neighbors=n_neighbors,
    mode='connectivity',
    include_self=True
)

assert np.allclose(graph.diagonal(), 1.0), "Diagonal should be 1.0"
print("  ✓ include_self=True")

# Test 5: include_self='auto' with connectivity
graph = kneighbors_graph_from_transformer(
    X,
    knn_transformer=KNeighborsTransformer,
    n_neighbors=n_neighbors,
    mode='connectivity',
    include_self='auto'
)

assert np.allclose(graph.diagonal(), 1.0), "Should include self for connectivity"
print("  ✓ include_self='auto' (connectivity)")

# Test 6: include_self='auto' with distance
graph = kneighbors_graph_from_transformer(
    X,
    knn_transformer=KNeighborsTransformer,
    n_neighbors=n_neighbors,
    mode='distance',
    include_self='auto'
)

assert np.allclose(graph.diagonal(), 0.0), "Should NOT include self for distance"
print("  ✓ include_self='auto' (distance)")

print("\n✓ All kneighbors_graph_from_transformer tests passed!")

Testing kneighbors_graph_from_transformer...
  ✓ Class with connectivity mode
  ✓ Class with distance mode
  ✓ Fitted transformer
  ✓ include_self=True
  ✓ include_self='auto' (connectivity)
  ✓ include_self='auto' (distance)

✓ All kneighbors_graph_from_transformer tests passed!


In [3]:
# Cell 3: Test brute_force_kneighbors_graph_from_rectangular_distance
print("Testing brute_force_kneighbors_graph_from_rectangular_distance...")

# Setup
np.random.seed(42)
square_dist = np.random.rand(10, 10)
np.fill_diagonal(square_dist, 0)
rect_dist = np.random.rand(10, 15)
n_neighbors = 3

# Test 1: Connectivity mode with square matrix
graph = brute_force_kneighbors_graph_from_rectangular_distance(
    square_dist,
    n_neighbors=n_neighbors,
    mode='connectivity',
    include_self=False
)

assert isinstance(graph, sps.csr_matrix), "Should return sparse matrix"
assert graph.shape == square_dist.shape, "Shape should match input"
assert set(graph.data) == {1.0}, "Values should be binary"
print("  ✓ Connectivity mode (square)")

# Test 2: Distance mode with square matrix
graph = brute_force_kneighbors_graph_from_rectangular_distance(
    square_dist,
    n_neighbors=n_neighbors,
    mode='distance',
    include_self=False
)

assert isinstance(graph, sps.csr_matrix), "Should return sparse matrix"
assert len(set(graph.data)) > 1, "Should contain actual distance values"
print("  ✓ Distance mode (square)")

# Test 3: Rectangular matrix
graph = brute_force_kneighbors_graph_from_rectangular_distance(
    rect_dist,
    n_neighbors=n_neighbors,
    mode='distance',
    include_self=False
)

assert graph.shape == rect_dist.shape, "Shape should match input"
print("  ✓ Rectangular matrix")

# Test 4: include_self adjustment
graph_with_self = brute_force_kneighbors_graph_from_rectangular_distance(
    square_dist,
    n_neighbors=n_neighbors,
    mode='connectivity',
    include_self=True
)

graph_without_self = brute_force_kneighbors_graph_from_rectangular_distance(
    square_dist,
    n_neighbors=n_neighbors - 1,
    mode='connectivity',
    include_self=False
)

assert np.allclose(
    np.array(graph_with_self.sum(axis=1)),
    np.array(graph_without_self.sum(axis=1))
), "include_self should adjust n_neighbors"
print("  ✓ include_self adjustment")

print("\n✓ All brute_force_kneighbors_graph tests passed!")

Testing brute_force_kneighbors_graph_from_rectangular_distance...
  ✓ Connectivity mode (square)
  ✓ Distance mode (square)
  ✓ Rectangular matrix
  ✓ include_self adjustment

✓ All brute_force_kneighbors_graph tests passed!


In [4]:
# Cell 3: Test brute_force_kneighbors_graph_from_rectangular_distance
print("Testing brute_force_kneighbors_graph_from_rectangular_distance...")

# Setup
np.random.seed(42)
square_dist = np.random.rand(10, 10)
np.fill_diagonal(square_dist, 0)
rect_dist = np.random.rand(10, 15)
n_neighbors = 3

# Test 1: Connectivity mode with square matrix
graph = brute_force_kneighbors_graph_from_rectangular_distance(
    square_dist,
    n_neighbors=n_neighbors,
    mode='connectivity',
    include_self=False
)

assert isinstance(graph, sps.csr_matrix), "Should return sparse matrix"
assert graph.shape == square_dist.shape, "Shape should match input"
assert set(graph.data) == {1.0}, "Values should be binary"
print("  ✓ Connectivity mode (square)")

# Test 2: Distance mode with square matrix
graph = brute_force_kneighbors_graph_from_rectangular_distance(
    square_dist,
    n_neighbors=n_neighbors,
    mode='distance',
    include_self=False
)

assert isinstance(graph, sps.csr_matrix), "Should return sparse matrix"
assert len(set(graph.data)) > 1, "Should contain actual distance values"
print("  ✓ Distance mode (square)")

# Test 3: Rectangular matrix
graph = brute_force_kneighbors_graph_from_rectangular_distance(
    rect_dist,
    n_neighbors=n_neighbors,
    mode='distance',
    include_self=False
)

assert graph.shape == rect_dist.shape, "Shape should match input"
print("  ✓ Rectangular matrix")

# Test 4: include_self adjustment
graph_with_self = brute_force_kneighbors_graph_from_rectangular_distance(
    square_dist,
    n_neighbors=n_neighbors,
    mode='connectivity',
    include_self=True
)

graph_without_self = brute_force_kneighbors_graph_from_rectangular_distance(
    square_dist,
    n_neighbors=n_neighbors - 1,
    mode='connectivity',
    include_self=False
)

assert np.allclose(
    np.array(graph_with_self.sum(axis=1)),
    np.array(graph_without_self.sum(axis=1))
), "include_self should adjust n_neighbors"
print("  ✓ include_self adjustment")

print("\n✓ All brute_force_kneighbors_graph tests passed!")

Testing brute_force_kneighbors_graph_from_rectangular_distance...
  ✓ Connectivity mode (square)
  ✓ Distance mode (square)
  ✓ Rectangular matrix
  ✓ include_self adjustment

✓ All brute_force_kneighbors_graph tests passed!


In [5]:
# Cell 5: Test convert_distance_matrix_to_kneighbors_matrix
print("Testing convert_distance_matrix_to_kneighbors_matrix...")

np.random.seed(42)
X = np.random.randn(10, 5)
dist_array = pairwise_distances(X, metric='euclidean')
dist_df = pd.DataFrame(
    dist_array,
    index=[f"s_{i}" for i in range(10)],
    columns=[f"s_{i}" for i in range(10)]
)
n_neighbors = 3

# Test 1: Basic conversion (array) - use symmetric=False
knn = convert_distance_matrix_to_kneighbors_matrix(
    dist_array,
    n_neighbors=n_neighbors,
    include_self=False,
    symmetric=False  # CHANGED: no symmetrization
)

assert knn.shape == dist_array.shape, "Shape should match input"
non_zero_counts = np.sum(knn > 0, axis=1)
assert np.all(non_zero_counts == n_neighbors), \
    "Each row should have exactly n_neighbors non-zero entries"
print("  ✓ Basic conversion (array)")

# Test 2: Basic conversion (DataFrame)
knn = convert_distance_matrix_to_kneighbors_matrix(
    dist_df,
    n_neighbors=n_neighbors,
    include_self=False,
    symmetric=False  # CHANGED: no symmetrization
)

assert isinstance(knn, pd.DataFrame), "Should return DataFrame"
assert list(knn.index) == list(dist_df.index), "Index should be preserved"
print("  ✓ Basic conversion (DataFrame)")

# Test 3: include_self
knn_with_self = convert_distance_matrix_to_kneighbors_matrix(
    dist_array,
    n_neighbors=n_neighbors,
    include_self=True,
    symmetric=False  # CHANGED
)

assert np.allclose(np.diag(knn_with_self), 0.0), \
    "Diagonal should be zero (nearest to self)"
print("  ✓ include_self parameter")

# Test 4: Symmetric
knn_sym = convert_distance_matrix_to_kneighbors_matrix(
    dist_array,
    n_neighbors=n_neighbors,
    symmetric=True
)

knn_asym = convert_distance_matrix_to_kneighbors_matrix(
    dist_array,
    n_neighbors=n_neighbors,
    symmetric=False
)

assert np.allclose(knn_sym, knn_sym.T), "Symmetric version should be symmetric"
assert np.sum(knn_sym > 0) >= np.sum(knn_asym > 0), \
    "Symmetrization should add entries"
print("  ✓ Symmetric parameter")

# Test 5: Condensed form
knn = convert_distance_matrix_to_kneighbors_matrix(
    dist_array,
    n_neighbors=n_neighbors,
    redundant_form=False,
    symmetric=False  # CHANGED
)

expected_len = 10 * 9 // 2
assert isinstance(knn, np.ndarray), "Should return ndarray"
assert len(knn) == expected_len, f"Length should be {expected_len}"
print("  ✓ Condensed form")

# Test 6: Values preserved
knn = convert_distance_matrix_to_kneighbors_matrix(
    dist_array,
    n_neighbors=n_neighbors,
    include_self=False,
    symmetric=False
)

# Check one row
i = 0
knn_dists = knn[i][knn[i] > 0]
orig_sorted = np.sort(dist_array[i])
expected_dists = orig_sorted[1:n_neighbors + 1]  # Skip diagonal

assert np.allclose(sorted(knn_dists), sorted(expected_dists)), \
    "Distance values should be preserved"
print("  ✓ Values preserved")

print("\n✓ All convert_distance_matrix_to_kneighbors_matrix tests passed!")

Testing convert_distance_matrix_to_kneighbors_matrix...
  ✓ Basic conversion (array)
  ✓ Basic conversion (DataFrame)
  ✓ include_self parameter
  ✓ Symmetric parameter
  ✓ Condensed form
  ✓ Values preserved

✓ All convert_distance_matrix_to_kneighbors_matrix tests passed!


In [6]:
# Cell 6: Test kneighbors_to_igraph (optional - requires igraph)
print("Testing kneighbors_to_igraph...")

try:
    import igraph as ig
    
    np.random.seed(42)
    n_samples = 10
    k = 3
    
    # Create mock kNN results
    D = np.random.rand(n_samples, k)
    I = np.zeros((n_samples, k), dtype=int)
    for i in range(n_samples):
        I[i] = np.random.choice(n_samples, k, replace=False)
    
    # Test 1: Basic conversion
    graph = kneighbors_to_igraph(D, I)
    assert isinstance(graph, ig.Graph), "Should return igraph.Graph"
    assert graph.is_directed(), "Should be directed"
    print("  ✓ Basic conversion")
    
    # Test 2: With custom index
    index = [f"node_{i}" for i in range(n_samples)]
    graph = kneighbors_to_igraph(D, I, index=index)
    assert isinstance(graph, ig.Graph), "Should return igraph.Graph"
    assert all(isinstance(v['name'], str) for v in graph.vs), \
        "Vertex names should be strings"
    print("  ✓ With custom index")
    
    # Test 3: include_self
    D_with_self = np.column_stack([np.ones(n_samples), D[:, :-1]])
    I_with_self = np.column_stack([np.arange(n_samples), I[:, :-1]])
    
    graph_with = kneighbors_to_igraph(D_with_self, I_with_self, include_self=True)
    graph_without = kneighbors_to_igraph(D_with_self, I_with_self, include_self=False)
    
    assert graph_without.ecount() < graph_with.ecount(), \
        "Graph without self should have fewer edges"
    print("  ✓ include_self parameter")
    
    print("\n✓ All kneighbors_to_igraph tests passed!")
    
except ImportError:
    print("  ⚠ igraph not installed - skipping tests")

Testing kneighbors_to_igraph...
  ✓ Basic conversion
  ✓ With custom index
  ✓ include_self parameter

✓ All kneighbors_to_igraph tests passed!


In [7]:
# Cell 7: Test KNeighborsCosineSimilarity - sklearn backend
print("Testing KNeighborsCosineSimilarity (sklearn backend)...")

np.random.seed(42)
n_samples = 100
n_features = 32
n_neighbors = 5

# Create L2-normalized data for cosine similarity
X = np.random.randn(n_samples, n_features).astype(np.float32)
norms = np.linalg.norm(X, axis=1, keepdims=True)
X = X / norms

# Test 1: Fit
knn = KNeighborsCosineSimilarity(n_neighbors=n_neighbors, backend='sklearn')
knn.fit(X)

assert hasattr(knn, 'n_samples_fit_'), "Should have n_samples_fit_"
assert hasattr(knn, 'n_features_in_'), "Should have n_features_in_"
assert knn.backend_ == 'sklearn', "Backend should be sklearn"
print("  ✓ Fit")

# Test 2: Transform
similarities, indices = knn.transform(X[:10])

assert similarities.shape == (10, n_neighbors), \
    f"Similarities shape should be (10, {n_neighbors})"
assert indices.shape == (10, n_neighbors), \
    f"Indices shape should be (10, {n_neighbors})"
assert np.all(similarities >= -1) and np.all(similarities <= 1), \
    "Similarities should be between -1 and 1"
print("  ✓ Transform")

# Test 3: fit_transform
knn = KNeighborsCosineSimilarity(n_neighbors=n_neighbors, backend='sklearn')
similarities, indices = knn.fit_transform(X)

assert similarities.shape == (n_samples, n_neighbors), \
    f"Similarities shape should be ({n_samples}, {n_neighbors})"
assert indices.shape == (n_samples, n_neighbors), \
    f"Indices shape should be ({n_samples}, {n_neighbors})"
print("  ✓ fit_transform")

# Test 4: to_igraph (optional)
try:
    import igraph as ig
    graph = knn.to_igraph()
    assert isinstance(graph, ig.Graph), "Should return igraph.Graph"
    assert graph.is_directed(), "Should be directed"
    print("  ✓ to_igraph")
    
    # Test 5: to_igraph with custom index
    index = [f"sample_{i}" for i in range(n_samples)]
    graph = knn.to_igraph(index=index)
    assert isinstance(graph, ig.Graph), "Should return igraph.Graph"
    print("  ✓ to_igraph with custom index")
    
except ImportError:
    print("  ⚠ igraph not installed - skipping to_igraph tests")

print("\n✓ All KNeighborsCosineSimilarity (sklearn) tests passed!")

Testing KNeighborsCosineSimilarity (sklearn backend)...
  ✓ Fit
  ✓ Transform
  ✓ fit_transform
  ✓ to_igraph
  ✓ to_igraph with custom index

✓ All KNeighborsCosineSimilarity (sklearn) tests passed!


In [8]:
# Cell 8: Test KNeighborsCosineSimilarity - FAISS backend (optional)
print("Testing KNeighborsCosineSimilarity (FAISS backend)...")

try:
    import faiss
    
    np.random.seed(42)
    n_samples = 100
    n_features = 32
    n_neighbors = 5
    
    # Create L2-normalized data
    X = np.random.randn(n_samples, n_features).astype(np.float32)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / norms
    
    # Test 1: Exact mode
    knn = KNeighborsCosineSimilarity(
        n_neighbors=n_neighbors,
        backend='faiss',
        mode='exact'
    )
    knn.fit(X)
    
    assert knn.backend_ == 'faiss', "Backend should be faiss"
    assert hasattr(knn, 'index_'), "Should have index_"
    
    similarities, indices = knn.transform(X[:10])
    assert similarities.shape == (10, n_neighbors), "Similarities shape correct"
    print("  ✓ FAISS exact mode")
    
    # Test 2: IVF mode
    knn = KNeighborsCosineSimilarity(
        n_neighbors=n_neighbors,
        backend='faiss',
        mode='ivf',
        n_voronoi_cells=10,
        n_probes=2
    )
    knn.fit(X)
    
    similarities, indices = knn.transform(X[:10])
    assert similarities.shape == (10, n_neighbors), "Similarities shape correct"
    print("  ✓ FAISS IVF mode")
    
    # Test 3: Auto backend (should use FAISS if available)
    knn = KNeighborsCosineSimilarity(n_neighbors=n_neighbors, backend='auto')
    knn.fit(X)
    
    assert knn.backend_ == 'faiss', "Auto should select FAISS when available"
    print("  ✓ Auto backend selection")
    
    print("\n✓ All KNeighborsCosineSimilarity (FAISS) tests passed!")
    
except ImportError:
    print("  ⚠ FAISS not installed - skipping FAISS tests")

Testing KNeighborsCosineSimilarity (FAISS backend)...
  ✓ FAISS exact mode
  ✓ FAISS IVF mode
  ✓ Auto backend selection

✓ All KNeighborsCosineSimilarity (FAISS) tests passed!


WARNING clustering 100 points to 10 centroids: please provide at least 390 training points


In [9]:
# Cell 9: Test edge cases
print("Testing edge cases...")

# Test 1: Single neighbor
X = np.random.randn(5, 3)
graph = kneighbors_graph_from_transformer(
    X,
    knn_transformer=KNeighborsTransformer,
    n_neighbors=1,
    mode='connectivity',
    include_self=False
)

assert np.all(np.array(graph.sum(axis=1)).ravel() == 1), \
    "Each row should have exactly 1 non-zero entry"
print("  ✓ Single neighbor")

# Test 2: All neighbors
X = np.random.randn(5, 3)
n_samples = X.shape[0]

graph = kneighbors_graph_from_transformer(
    X,
    knn_transformer=KNeighborsTransformer,
    n_neighbors=n_samples - 1,
    mode='connectivity',
    include_self=False
)

dense = graph.todense()
assert np.sum(dense > 0) == n_samples * (n_samples - 1), \
    "Should be fully connected except diagonal"
print("  ✓ All neighbors")

# Test 3: Identical points
X = np.ones((5, 3))
dists = pairwise_distances_kneighbors(
    X,
    metric='euclidean',
    n_neighbors=2
)

assert np.allclose(dists, 0), "All pairwise distances should be zero"
print("  ✓ Identical points")

print("\n✓ All edge case tests passed!")

Testing edge cases...
  ✓ Single neighbor
  ✓ All neighbors
  ✓ Identical points

✓ All edge case tests passed!


In [10]:
# Cell 10: Summary
print("\n" + "="*60)
print("TEST SUMMARY")
print("="*60)
print("✓ kneighbors_graph_from_transformer")
print("✓ brute_force_kneighbors_graph_from_rectangular_distance")
print("✓ pairwise_distances_kneighbors")
print("✓ convert_distance_matrix_to_kneighbors_matrix")
print("✓ kneighbors_to_igraph (if igraph installed)")
print("✓ KNeighborsCosineSimilarity (sklearn backend)")
print("✓ KNeighborsCosineSimilarity (FAISS backend, if installed)")
print("✓ Edge cases")
print("="*60)
print("All tests completed successfully! 🎉")


TEST SUMMARY
✓ kneighbors_graph_from_transformer
✓ brute_force_kneighbors_graph_from_rectangular_distance
✓ pairwise_distances_kneighbors
✓ convert_distance_matrix_to_kneighbors_matrix
✓ kneighbors_to_igraph (if igraph installed)
✓ KNeighborsCosineSimilarity (sklearn backend)
✓ KNeighborsCosineSimilarity (FAISS backend, if installed)
✓ Edge cases
All tests completed successfully! 🎉


In [2]:
# %% Test FaissKNNClassifier and FaissKNNTransformer
import time
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.neighbors import (
    KNeighborsTransformer,
    KNeighborsClassifier,
)
from sklearn.base import clone
from skclust.neighbors import FaissKNNClassifier, FaissKNNTransformer

n_samples, n_features, n_neighbors = 5000, 50, 10
X, y = make_blobs(n_samples=n_samples, n_features=n_features, centers=8, random_state=42)
X = X.astype(np.float32)
X_new = np.random.RandomState(99).randn(500, n_features).astype(np.float32)

# ══════════════════════════════════════════════════════════════════════════════
# 1. Classifier
# ══════════════════════════════════════════════════════════════════════════════

# ── sklearn baseline ──
t0 = time.perf_counter()
sk_clf = KNeighborsClassifier(n_neighbors=n_neighbors, metric="euclidean")
sk_clf.fit(X, y)
sk_preds_train = sk_clf.predict(X)
sk_preds_new = sk_clf.predict(X_new)
sk_clf_time = time.perf_counter() - t0

# ── FAISS ──
t0 = time.perf_counter()
faiss_clf = FaissKNNClassifier(n_neighbors=n_neighbors)
faiss_clf.fit(X, y)
faiss_preds_train = faiss_clf.predict(X)
faiss_preds_new = faiss_clf.predict(X_new)
faiss_clf_time = time.perf_counter() - t0

# Prediction agreement
train_agreement = (faiss_preds_train == sk_preds_train).mean()
new_agreement = (faiss_preds_new == sk_preds_new).mean()
assert train_agreement > 0.9, f"Train agreement too low: {train_agreement:.2f}"
assert new_agreement > 0.9, f"New data agreement too low: {new_agreement:.2f}"

# kneighbors shapes
distances, indices = faiss_clf.kneighbors(X)
assert distances.shape == (n_samples, n_neighbors)
assert indices.shape == (n_samples, n_neighbors)
assert (distances >= 0).all()

distances_3, indices_3 = faiss_clf.kneighbors(X[:10], n_neighbors=3)
assert distances_3.shape == (10, 3)

# sklearn API
assert clone(faiss_clf).n_neighbors == n_neighbors

print(f"✓ FaissKNNClassifier")
print(f"  Train agreement: {train_agreement:.1%} | Unseen agreement: {new_agreement:.1%}")
print(f"  Timing — sklearn: {sk_clf_time:.3f}s | FAISS: {faiss_clf_time:.3f}s | speedup: {sk_clf_time/faiss_clf_time:.1f}x")

# ══════════════════════════════════════════════════════════════════════════════
# 2. Transformer
# ══════════════════════════════════════════════════════════════════════════════

# ── sklearn baseline ──
t0 = time.perf_counter()
sk_trans = KNeighborsTransformer(n_neighbors=n_neighbors, mode="distance", metric="euclidean")
sk_dist_train = sk_trans.fit_transform(X)
sk_dist_new = sk_trans.transform(X_new)
sk_trans_time = time.perf_counter() - t0

# ── FAISS ──
t0 = time.perf_counter()
faiss_trans = FaissKNNTransformer(n_neighbors=n_neighbors)
faiss_dist_train = faiss_trans.fit_transform(X)
faiss_dist_new = faiss_trans.transform(X_new)
faiss_trans_time = time.perf_counter() - t0

# Shape and sparsity
assert faiss_dist_train.shape == (n_samples, n_samples)
assert faiss_dist_new.shape == (500, n_samples)
assert 0 < faiss_dist_train.nnz <= n_samples * n_neighbors
assert (faiss_dist_train.data >= 0).all()
assert (faiss_dist_new.data >= 0).all()

# Neighbor overlap
train_overlaps = []
for i in range(n_samples):
    faiss_nn = set(faiss_dist_train.getrow(i).indices)
    sk_nn = set(sk_dist_train.getrow(i).indices)
    if sk_nn:
        train_overlaps.append(len(faiss_nn & sk_nn) / len(sk_nn))
train_overlap = np.mean(train_overlaps)
assert train_overlap > 0.8, f"Train neighbor overlap too low: {train_overlap:.2f}"

# Distance scale check: compare sorted per-row distances
# (avoids sparse matrix indexing issues with mismatched sparsity patterns)
faiss_median_dists = []
sk_median_dists = []
for i in range(min(500, n_samples)):
    faiss_row = faiss_dist_train.getrow(i)
    sk_row = sk_dist_train.getrow(i)
    if faiss_row.nnz > 0 and sk_row.nnz > 0:
        faiss_median_dists.append(np.median(faiss_row.data))
        sk_median_dists.append(np.median(sk_row.data))

faiss_median_dists = np.array(faiss_median_dists)
sk_median_dists = np.array(sk_median_dists)
distance_ratio = np.median(faiss_median_dists / sk_median_dists)
assert 0.9 < distance_ratio < 1.1, (
    f"Distance scale mismatch: FAISS/sklearn ratio = {distance_ratio:.3f}. "
    f"If ~sqrt(d) this suggests sqrt is being applied twice; "
    f"if ~d this suggests squared L2 not being converted."
)

# sklearn API
assert clone(faiss_trans).n_neighbors == n_neighbors

print(f"\n✓ FaissKNNTransformer")
print(f"  Neighbor overlap: {train_overlap:.1%}")
print(f"  Distance ratio (FAISS/sklearn): {distance_ratio:.4f}")
print(f"  Timing — sklearn: {sk_trans_time:.3f}s | FAISS: {faiss_trans_time:.3f}s | speedup: {sk_trans_time/faiss_trans_time:.1f}x")

# ── 3. Unseen data ──
assert faiss_trans.transform(X_new).shape == (500, n_samples)
assert len(faiss_clf.predict(X_new)) == 500

print(f"\n✓ Unseen data: pass")
print("\nAll tests passed.")

✓ FaissKNNClassifier
  Train agreement: 100.0% | Unseen agreement: 100.0%
  Timing — sklearn: 0.039s | FAISS: 0.014s | speedup: 2.8x

✓ FaissKNNTransformer
  Neighbor overlap: 90.9%
  Distance ratio (FAISS/sklearn): 0.9963
  Timing — sklearn: 0.037s | FAISS: 0.009s | speedup: 4.0x

✓ Unseen data: pass

All tests passed.
